# Notebook 1 — Quantize Qwen2.5-3B-Instruct with AWQ and upload to Hugging Face

**Target:** Google Colab with an NVIDIA T4 GPU.

This notebook quantizes `Qwen/Qwen2.5-3B-Instruct` to 4-bit AWQ (`W4A16_ASYM`) using LLM Compressor, saves the **compressed-tensors** checkpoint, and uploads it to your Hugging Face Hub repository. By default, the new Hub repository is private.

## Compatibility strategy (checked 14 October 2026)

| Component | Pinned version | Reason |
|---|---|---|
| Python | 3.12 in an isolated `uv` environment | Does not depend on Colab's changing base Python environment |
| LLM Compressor | `0.9.0` | AWQ recipe supported; release pins `compressed-tensors==0.13.0` |
| PyTorch for quantization | `2.9.1` with CUDA 12.8 wheels | Within LLM Compressor 0.9.0's declared Torch range |
| Transformers / datasets / accelerate | `4.57.3` / `4.4.1` / `1.12.0` | Within LLM Compressor 0.9.0's release bounds |
| Hugging Face Hub | `0.36.2` | Keeps this older Transformers environment on the Hub 0.x API |
| Output format | `compressed-tensors`, AWQ `W4A16_ASYM` | Matched by the serving notebook's vLLM release |

**Why pin an older pair?** vLLM `0.18.0` explicitly pins `compressed-tensors==0.13.0`, and LLM Compressor `0.9.0` release metadata pins the same version. A reported installation issue shows that installing a mismatched vLLM can downgrade `compressed-tensors` and break the quantization environment. We isolate the two toolchains and never install vLLM into this environment. See [LLM Compressor 0.9.0 dependencies](https://github.com/vllm-project/llm-compressor/blob/0.9.0/setup.py), [vLLM 0.18.0 dependencies](https://github.com/vllm-project/vllm/blob/v0.18.0/requirements/common.txt), and [the reported version-conflict issue](https://github.com/vllm-project/llm-compressor/issues/2258).

This is a pinned, compatibility-first workflow rather than the newest possible stack. It includes runtime checks, but has not been executed on a live Colab T4 from this chat.

## 1. Select and verify the Colab GPU

In Colab, select **Runtime → Change runtime type → T4 GPU**, connect to a fresh runtime, and run the next cell. It checks the actual GPU rather than assuming Colab assigned a T4.

In [1]:
import os, sys, subprocess, platform

print("Base Python:", sys.version)
print("Platform:", platform.platform())
subprocess.run(["nvidia-smi"], check=False)

import torch
print("Colab base Torch:", torch.__version__)
print("Base Torch CUDA:", torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select Runtime > Change runtime type > T4 GPU, reconnect, and rerun.")
name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU: {name}; capability: {capability}; VRAM: {vram_gib:.1f} GiB")
if capability < (7, 5):
    raise RuntimeError(f"This workflow expects compute capability >= 7.5; found {capability}.")

Base Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
Colab base Torch: 2.11.0+cu130
Base Torch CUDA: 13.0
GPU: Tesla T4; capability: (7, 5); VRAM: 14.6 GiB


## 2. Set the model repository and Hugging Face credentials

Create a Hugging Face access token with **write** permission. The token is entered through hidden input and is not saved into the notebook source. This notebook creates a private model repository by default.

In [2]:
from getpass import getpass

BASE_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
REPO_ID = input("Target Hub repo (e.g. your-username/qwen2.5-3b-instruct-awq-w4a16): ").strip()
if "/" not in REPO_ID or REPO_ID.startswith("YOUR_"):
    raise ValueError("Enter a real Hugging Face repo ID in username/repo-name format.")
HF_TOKEN = getpass("Hugging Face write token (hidden): ").strip()
if not HF_TOKEN.startswith("hf_"):
    raise ValueError("A Hugging Face token beginning with 'hf_' is required.")

# Keep the repo private by default. Change to False only if you intentionally want a public repo.
REPO_PRIVATE = True
CALIBRATION_SAMPLES = 64
MAX_SEQ_LENGTH = 512

print("Base model:", BASE_MODEL_ID)
print("Destination:", REPO_ID)
print("Private repository:", REPO_PRIVATE)
print("Calibration samples:", CALIBRATION_SAMPLES, "| max sequence length:", MAX_SEQ_LENGTH)

Target Hub repo (e.g. your-username/qwen2.5-3b-instruct-awq-w4a16): abdelrahman-omran/qwen2.5-3b-instruct-awq-w4a16
Hugging Face write token (hidden): ··········
Base model: Qwen/Qwen2.5-3B-Instruct
Destination: abdelrahman-omran/qwen2.5-3b-instruct-awq-w4a16
Private repository: True
Calibration samples: 64 | max sequence length: 512


## 3. Create an isolated quantization environment

The base Colab Torch is left alone. LLM Compressor gets its own Python 3.12 environment with the pinned CUDA/PyTorch stack. This avoids mixing the quantizer with vLLM dependencies or upgrading Colab’s preinstalled Torch.

In [3]:
from pathlib import Path
import os, subprocess, sys

ROOT = Path("/content/qwen25-awq-quantization")
ENV_PATH = ROOT / "venv"
QUANT_PY = ENV_PATH / "bin" / "python"
MODEL_DIR = Path("/content/Qwen2.5-3B-Instruct-AWQ-W4A16")
PYTHON_VERSION = "3.12"

ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "python", "install", PYTHON_VERSION], check=True)
if not QUANT_PY.exists():
    subprocess.run(["uv", "venv", "--python", PYTHON_VERSION, "--managed-python", str(ENV_PATH)], check=True)
print("Quantization Python:", QUANT_PY)
print("Output folder:", MODEL_DIR)

Quantization Python: /content/qwen25-awq-quantization/venv/bin/python
Output folder: /content/Qwen2.5-3B-Instruct-AWQ-W4A16


## 4. Install and verify the pinned quantization stack

Do not install vLLM in this environment. `uv pip check` and explicit version assertions must pass before quantization begins.

In [4]:
subprocess.run([
    "uv", "pip", "install", "--python", str(QUANT_PY), "--torch-backend=cu128",
    "torch==2.9.1", "torchaudio==2.9.1", "torchvision==0.24.1",
    "llmcompressor==0.9.0", "compressed-tensors==0.13.0",
    "transformers==4.57.3", "datasets==4.4.1", "accelerate==1.12.0",
    "huggingface-hub==0.36.2",
], check=True)
subprocess.run(["uv", "pip", "check", "--python", str(QUANT_PY)], check=True)

inspect_script = r"""import sys, torch
from importlib.metadata import version
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__, "CUDA build:", torch.version.cuda)
for package in ("llmcompressor", "compressed-tensors", "transformers", "datasets", "accelerate", "huggingface-hub"):
    print(f"{package}:", version(package))
assert version("llmcompressor") == "0.9.0"
assert version("compressed-tensors") == "0.13.0"
assert version("transformers") == "4.57.3"
assert version("datasets") == "4.4.1"
assert version("accelerate") == "1.12.0"
assert version("huggingface-hub") == "0.36.2"
assert version("torch").split("+")[0] == "2.9.1"
assert torch.cuda.is_available(), "Quantization environment cannot see the CUDA GPU"
"""
subprocess.run([str(QUANT_PY), "-c", inspect_script], check=True)

CompletedProcess(args=['/content/qwen25-awq-quantization/venv/bin/python', '-c', 'import sys, torch\nfrom importlib.metadata import version\nprint("Python:", sys.version.split()[0])\nprint("Torch:", torch.__version__, "CUDA build:", torch.version.cuda)\nfor package in ("llmcompressor", "compressed-tensors", "transformers", "datasets", "accelerate", "huggingface-hub"):\n    print(f"{package}:", version(package))\nassert version("llmcompressor") == "0.9.0"\nassert version("compressed-tensors") == "0.13.0"\nassert version("transformers") == "4.57.3"\nassert version("datasets") == "4.4.1"\nassert version("accelerate") == "1.12.0"\nassert version("huggingface-hub") == "0.36.2"\nassert version("torch").split("+")[0] == "2.9.1"\nassert torch.cuda.is_available(), "Quantization environment cannot see the CUDA GPU"\n'], returncode=0)

## 5. Run AWQ quantization

The notebook uses 64 calibration samples with a 512-token cap as a conservative starting point for the T4. Quantization can still take time and use substantial VRAM. If it runs out of memory, try 32 samples and a 256-token cap before increasing either value.

In [ ]:
if MODEL_DIR.exists() and any(MODEL_DIR.iterdir()):
    raise FileExistsError(
        f"{MODEL_DIR} already contains files. Use a fresh Colab runtime or change MODEL_DIR before rerunning."
    )

QUANT_SCRIPT = ROOT / "quantize_awq.py"
quant_script = r"""import os, torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from llmcompressor import oneshot
from llmcompressor.modifiers.awq import AWQModifier

model_id = os.environ["BASE_MODEL_ID"]
output_dir = os.environ["MODEL_DIR"]
num_samples = int(os.environ["CALIBRATION_SAMPLES"])
max_seq_length = int(os.environ["MAX_SEQ_LENGTH"])

torch.set_num_threads(4)
print(f"Loading {model_id} in FP16 for AWQ calibration...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading calibration samples...")
ds = load_dataset(
    "HuggingFaceH4/ultrachat_200k",
    split=f"train_sft[:{max(num_samples, 128)}]",
).shuffle(seed=42)

def apply_chat_template(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}

ds = ds.map(apply_chat_template)

def tokenize(example):
    return tokenizer(
        example["text"],
        padding=False,
        max_length=max_seq_length,
        truncation=True,
        add_special_tokens=False,
    )
ds = ds.map(tokenize, remove_columns=ds.column_names)

recipe = [AWQModifier(ignore=["lm_head"], scheme="W4A16_ASYM", targets=["Linear"])]
print("Starting one-shot AWQ quantization...")
oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=max_seq_length,
    num_calibration_samples=num_samples,
)
os.makedirs(output_dir, exist_ok=True)
model.save_pretrained(output_dir, save_compressed=True)
tokenizer.save_pretrained(output_dir)
print("Saved compressed checkpoint to:", output_dir)
print("Quantization completed.")
"""
QUANT_SCRIPT.write_text(quant_script)
quant_env = os.environ.copy()
quant_env.update({
    "BASE_MODEL_ID": BASE_MODEL_ID,
    "MODEL_DIR": str(MODEL_DIR),
    "CALIBRATION_SAMPLES": str(CALIBRATION_SAMPLES),
    "MAX_SEQ_LENGTH": str(MAX_SEQ_LENGTH),
    "HF_TOKEN": HF_TOKEN,
    "TOKENIZERS_PARALLELISM": "false",
})

config_path = MODEL_DIR / "config.json"


## duo_scaling=False


In [7]:

from pathlib import Path
import subprocess

script_text = QUANT_SCRIPT.read_text()

old = (
    'recipe = [AWQModifier(ignore=["lm_head"], '
    'scheme="W4A16_ASYM", targets=["Linear"])]'
)

new = (
    'recipe = [AWQModifier(ignore=["lm_head"], '
    'scheme="W4A16_ASYM", targets=["Linear"], '
    'duo_scaling=False)]'
)

if old in script_text:
    script_text = script_text.replace(old, new, 1)
    QUANT_SCRIPT.write_text(script_text)
    print("Patched AWQ: duo_scaling=False")
elif new in script_text:
    print("The AWQ workaround is already enabled.")
else:
    raise RuntimeError(
        "Expected AWQ recipe not found. Inspect QUANT_SCRIPT before proceeding."
    )

if MODEL_DIR.exists() and any(MODEL_DIR.iterdir()):
    raise FileExistsError(
        f"{MODEL_DIR} is not empty. "
        "Use a fresh output directory to avoid overwriting files."
    )

# Rerun with the existing, pinned quantization environment.
subprocess.run(
    [str(QUANT_PY), "-u", str(QUANT_SCRIPT)],
    env=quant_env,
    check=True,
)

Patched AWQ: duo_scaling=False


CompletedProcess(args=['/content/qwen25-awq-quantization/venv/bin/python', '-u', '/content/qwen25-awq-quantization/quantize_awq.py'], returncode=0)

## 6. Validate the checkpoint format before upload

The serving notebook expects the **compressed-tensors** format, not a folder produced by AutoAWQ with different metadata.

In [8]:
import json

model_config = json.loads((MODEL_DIR / "config.json").read_text())
quant_config = model_config.get("quantization_config", {})
quant_method = quant_config.get("quant_method", quant_config.get("quantization_method", "not found"))
print("Quantization config:", quant_config)
print("Quantization method:", quant_method)
if quant_method not in ("compressed-tensors", "compressed_tensors"):
    raise RuntimeError(f"Expected compressed-tensors metadata, got {quant_method!r}; do not upload/serve yet.")
print("Checkpoint metadata validation passed.")

Quantization config: {'config_groups': {'group_0': {'format': 'pack-quantized', 'input_activations': None, 'output_activations': None, 'targets': ['Linear'], 'weights': {'actorder': None, 'block_structure': None, 'dynamic': False, 'group_size': 128, 'num_bits': 4, 'observer': 'minmax', 'observer_kwargs': {}, 'scale_dtype': None, 'strategy': 'group', 'symmetric': False, 'type': 'int', 'zp_dtype': 'torch.int8'}}}, 'format': 'pack-quantized', 'global_compression_ratio': None, 'ignore': ['lm_head'], 'kv_cache_scheme': None, 'quant_method': 'compressed-tensors', 'quantization_status': 'compressed', 'sparsity_config': {}, 'transform_config': {}, 'version': '0.13.0'}
Quantization method: compressed-tensors
Checkpoint metadata validation passed.


## 7. Log in and upload the model to Hugging Face

This authenticates with the hidden token, creates the destination as a private repo by default, adds a short model card, and uploads all checkpoint and tokenizer files. Keep the write token private. The token is used through an environment variable and is not written into the notebook file.

In [9]:
README = MODEL_DIR / "README.md"
README.write_text(f"""---
license: apache-2.0
base_model: {BASE_MODEL_ID}
tags:
  - awq
  - compressed-tensors
  - vllm
  - text-generation
---

# Qwen2.5-3B-Instruct AWQ W4A16

This repository contains an AWQ W4A16 compressed-tensors checkpoint derived from `{BASE_MODEL_ID}`.

- Quantization: AWQ `W4A16_ASYM` using LLM Compressor 0.9.0
- Checkpoint format: `compressed-tensors`
- Calibration: `{CALIBRATION_SAMPLES}` samples from `HuggingFaceH4/ultrachat_200k`
- Calibration sequence cap: `{MAX_SEQ_LENGTH}` tokens
- Intended serving engine: vLLM with a compatible `compressed-tensors` version

Please review the original model card and license before redistributing this derivative.
""")

upload_script = r"""import os
from huggingface_hub import HfApi, login

token = os.environ["HF_TOKEN"]
repo_id = os.environ["HF_REPO_ID"]
folder = os.environ["MODEL_DIR"]
private = os.environ.get("HF_REPO_PRIVATE", "true").lower() == "true"

login(token=token, add_to_git_credential=False)
api = HfApi(token=token)
identity = api.whoami()
print("Authenticated to Hugging Face as:", identity.get("name", "(name unavailable)"))
api.create_repo(repo_id=repo_id, repo_type="model", private=private, exist_ok=True)
api.upload_folder(
    repo_id=repo_id,
    repo_type="model",
    folder_path=folder,
    commit_message="Upload Qwen2.5-3B-Instruct AWQ W4A16 compressed-tensors checkpoint",
)
print("Upload finished:", f"https://huggingface.co/{repo_id}")
print("Repository privacy:", "private" if private else "public")
"""
UPLOAD_SCRIPT = ROOT / "upload_to_hub.py"
UPLOAD_SCRIPT.write_text(upload_script)
upload_env = os.environ.copy()
upload_env.update({
    "HF_TOKEN": HF_TOKEN,
    "HF_REPO_ID": REPO_ID,
    "HF_REPO_PRIVATE": str(REPO_PRIVATE).lower(),
    "MODEL_DIR": str(MODEL_DIR),
})
subprocess.run([str(QUANT_PY), str(UPLOAD_SCRIPT)], env=upload_env, check=True)
print("Next: open Notebook 2, enter this Hub repository ID, and serve it with vLLM.")

Next: open Notebook 2, enter this Hub repository ID, and serve it with vLLM.


## Troubleshooting notes

- **CUDA out of memory during quantization:** try `CALIBRATION_SAMPLES = 32` and `MAX_SEQ_LENGTH = 256`; restart with a clean runtime if VRAM remains allocated.
- **Dependency conflict:** stop if `uv pip check` or a version assertion fails. Do not install vLLM or upgrade Torch in this environment.
- **Hub authorization error:** ensure the token has write permission and the repo ID uses `username/repo-name`.
- **Upload interrupted:** rerun the upload step after revalidating the output folder; `upload_folder` can upload to an existing repo.
- **Never paste the token into a code cell literal or commit it to a repository.**